In [4]:
import pandas as pd

data = pd.read_json('CrawlerHandball/bundesliga_data_filtered.jsonl', lines=True)
data.head()

,match_id,datum_zapasu,domaci_tym,hoste_tym,goly_domaci,goly_hoste,kurz_domaci,kurz_remiza,kurz_hoste,domaci_hraci,hoste_hraci
0,14211269,27.03.2026,MT Melsungen,THW Kiel,30,29,3.60,8.5,1.50,"[{'jmeno': 'Kristof Palasics', 'uspesnost_zakr...","[{'jmeno': 'Domagoj Duvnjak', 'goly': 3}, {'jm..."
1,14211229,27.12.2025,ThSV Eisenach,SC Magdeburg,25,30,10.00,15.0,1.12,"[{'jmeno': 'Silvio Heinevetter', 'uspesnost_za...","[{'jmeno': 'Tim Hornke', 'goly': 0}, {'jmeno':..."
2,14211136,23.12.2025,SC Magdeburg,THW Kiel,26,26,1.22,11.0,6.50,"[{'jmeno': 'Tim Hornke', 'goly': 1}, {'jmeno':...","[{'jmeno': 'Domagoj Duvnjak', 'goly': 0}, {'jm..."
3,14211211,14.12.2025,TSV Hannover Burgdorf,TVB 1898 Stuttgart,28,22,1.52,8.5,3.40,"[{'jmeno': 'Marius Steinhauser', 'goly': 2}, {...","[{'jmeno': 'Kai Hafner', 'goly': 4}, {'jmeno':..."
4,14211192,29.11.2025,SG Flensburg Handewitt,VfL Gummersbach,37,37,1.32,10.0,5.00,"[{'jmeno': 'Kent Robin Tonnesen', 'goly': 5}, ...","[{'jmeno': 'Kentin Mahe', 'goly': 0}, {'jmeno'..."


In [5]:
data['datum_zapasu'] = pd.to_datetime(data['datum_zapasu'], format='%d.%m.%Y')
data = data.sort_values('datum_zapasu')
data = data.reset_index(drop=True)
data.tail(10)

,match_id,datum_zapasu,domaci_tym,hoste_tym,goly_domaci,goly_hoste,kurz_domaci,kurz_remiza,kurz_hoste,domaci_hraci,hoste_hraci
2737,14211267,2026-03-15,SC DHfK Leipzig,TVB 1898 Stuttgart,29,29,1.75,8.0,2.70,"[{'jmeno': 'Tomas Mrkva', 'uspesnost_zakroku_p...","[{'jmeno': 'Kai Hafner', 'goly': 5}, {'jmeno':..."
2738,14203131,2026-03-26,HSV Hamburg,Bergischer HC,34,35,1.32,10.0,5.00,"[{'jmeno': 'Azat Valiullin', 'goly': 0}, {'jme...","[{'jmeno': 'Christopher Rudeck', 'uspesnost_za..."
2739,14211270,2026-03-27,SG Flensburg Handewitt,TSV GWD Minden,35,31,1.02,25.0,22.00,"[{'jmeno': 'Kent Robin Tonnesen', 'goly': 3}, ...","[{'jmeno': 'Malte Semisch', 'uspesnost_zakroku..."
2740,14211269,2026-03-27,MT Melsungen,THW Kiel,30,29,3.60,8.5,1.50,"[{'jmeno': 'Kristof Palasics', 'uspesnost_zakr...","[{'jmeno': 'Domagoj Duvnjak', 'goly': 3}, {'jm..."
2741,14211272,2026-03-28,SC Magdeburg,Füchse Berlin,35,33,1.78,8.0,2.70,"[{'jmeno': 'Tim Hornke', 'goly': 3}, {'jmeno':...","[{'jmeno': 'Aitor Arino', 'goly': 0}, {'jmeno'..."
2742,14211273,2026-03-28,Rhein Neckar Löwen,HSG Wetzlar,41,27,1.20,12.0,7.50,"[{'jmeno': 'Patrick Groetzki', 'goly': 4}, {'j...","[{'jmeno': 'Andreas Palicka', 'uspesnost_zakro..."
2743,14211274,2026-03-28,VfL Gummersbach,TSV Hannover Burgdorf,33,27,1.37,9.5,4.50,"[{'jmeno': 'Kentin Mahe', 'goly': 0}, {'jmeno'...","[{'jmeno': 'Marius Steinhauser', 'goly': 2}, {..."
2744,14203134,2026-03-29,TVB 1898 Stuttgart,TBV Lemgo,32,32,3.50,8.5,1.52,"[{'jmeno': 'Kai Hafner', 'goly': 8}, {'jmeno':...","[{'jmeno': 'Bobby Schagen', 'goly': 3}, {'jmen..."
2745,14211276,2026-03-29,Frisch Auf Göppingen,HC Erlangen,27,24,1.67,8.0,3.00,"[{'jmeno': 'Marcel Schiller', 'goly': 4}, {'jm...","[{'jmeno': 'Dario Quenstedt', 'uspesnost_zakro..."
2746,14211275,2026-03-29,SC DHfK Leipzig,ThSV Eisenach,29,29,1.80,8.0,2.70,"[{'jmeno': 'Tomas Mrkva', 'uspesnost_zakroku_p...","[{'jmeno': 'Silvio Heinevetter', 'uspesnost_za..."


In [6]:
def forma_tymu(row, data, nazev_tymu):
    aktualni_datum = row['datum_zapasu']
    tym = row[nazev_tymu]
    historie = data[data['datum_zapasu'] < aktualni_datum]

    zapasy_tymu = historie[(historie['domaci_tym'] == tym) | (historie['hoste_tym'] == tym)]

    posledni_zapasy = zapasy_tymu.tail(5)

    if len(posledni_zapasy) == 0:
        return 0.5

    vyhry = 0
    for _, zapas in posledni_zapasy.iterrows():
        if zapas['domaci_tym'] == tym and zapas['goly_domaci'] > zapas['goly_hoste']:
            vyhry += 1
        elif zapas['hoste_tym'] == tym and zapas['goly_hoste'] > zapas['goly_domaci']:
            vyhry += 1

    return vyhry / len(posledni_zapasy)

In [7]:
data['forma_tymu_domaci'] = data.apply(lambda row: forma_tymu(row, data, 'domaci_tym'), axis=1)
data['forma_tymu_hoste'] = data.apply(lambda row: forma_tymu(row, data, 'hoste_tym'), axis=1)

In [8]:
data.tail(20)

,match_id,datum_zapasu,domaci_tym,hoste_tym,goly_domaci,goly_hoste,kurz_domaci,kurz_remiza,kurz_hoste,domaci_hraci,hoste_hraci,forma_tymu_domaci,forma_tymu_hoste
2727,14211263,2026-03-08,HSG Wetzlar,MT Melsungen,32,38,3.80,8.5,1.47,"[{'jmeno': 'Andreas Palicka', 'uspesnost_zakro...","[{'jmeno': 'Timo Kastening', 'goly': 6}, {'jme...",0.0,0.6
2728,14211268,2026-03-09,Frisch Auf Göppingen,ThSV Eisenach,26,23,1.70,8.0,3.00,"[{'jmeno': 'Marcel Schiller', 'goly': 4}, {'jm...","[{'jmeno': 'Silvio Heinevetter', 'uspesnost_za...",0.2,0.2
2729,14211260,2026-03-13,TSV Hannover Burgdorf,HSV Hamburg,35,37,1.50,8.5,3.50,"[{'jmeno': 'Marius Steinhauser', 'goly': 7}, {...","[{'jmeno': 'Casper Mortensen', 'goly': 0}, {'j...",0.4,0.2
2730,14211271,2026-03-13,ThSV Eisenach,Rhein Neckar Löwen,29,29,3.20,8.0,1.57,"[{'jmeno': 'Silvio Heinevetter', 'uspesnost_za...","[{'jmeno': 'Patrick Groetzki', 'goly': 0}, {'j...",0.2,0.6
2731,14211262,2026-03-13,MT Melsungen,VfL Gummersbach,24,31,2.20,7.5,2.10,"[{'jmeno': 'Timo Kastening', 'goly': 1}, {'jme...","[{'jmeno': 'Kentin Mahe', 'goly': 0}, {'jmeno'...",0.6,1.0
2732,14203247,2026-03-14,TSV GWD Minden,HSG Wetzlar,31,33,2.70,7.5,1.80,"[{'jmeno': 'Malte Semisch', 'uspesnost_zakroku...","[{'jmeno': 'Andreas Palicka', 'uspesnost_zakro...",0.0,0.0
2733,14211264,2026-03-14,THW Kiel,SG Flensburg Handewitt,37,33,1.75,8.0,2.70,"[{'jmeno': 'Domagoj Duvnjak', 'goly': 0}, {'jm...","[{'jmeno': 'Kent Robin Tonnesen', 'goly': 0}, ...",0.6,0.8
2734,14211266,2026-03-15,TBV Lemgo,Frisch Auf Göppingen,27,30,1.15,13.0,9.00,"[{'jmeno': 'Bobby Schagen', 'goly': 1}, {'jmen...","[{'jmeno': 'Marcel Schiller', 'goly': 2}, {'jm...",0.4,0.4
2735,14211265,2026-03-15,Füchse Berlin,HC Erlangen,45,29,1.06,20.0,15.00,"[{'jmeno': 'Aitor Arino', 'goly': 0}, {'jmeno'...","[{'jmeno': 'Sebastian Firnhaber', 'goly': 2}, ...",1.0,0.2
2736,14203128,2026-03-15,Bergischer HC,SC Magdeburg,25,27,14.00,18.0,1.07,"[{'jmeno': 'Christopher Rudeck', 'uspesnost_za...","[{'jmeno': 'Tim Hornke', 'goly': 0}, {'jmeno':...",0.2,0.8


In [9]:
def prum_vstrelenych_obdrzenych(row, data, nazev_tymu):
    aktualni_datum = row['datum_zapasu']
    tym = row[nazev_tymu]

    historie = data[data['datum_zapasu'] < aktualni_datum]
    zapasy_tymu = historie[(historie['domaci_tym'] == tym) | (historie['hoste_tym'] == tym)]

    poslednich_5 = zapasy_tymu.tail(5)

    if len(poslednich_5) == 0:
        return 10.0, 20.0

    vstrelene_goly = 0
    obdrzene_goly = 0

    for _, zapas in poslednich_5.iterrows():
        if zapas['domaci_tym'] == tym:
            vstrelene_goly += zapas['goly_domaci']
            obdrzene_goly += zapas['goly_hoste']
        else:
            vstrelene_goly += zapas['goly_hoste']
            obdrzene_goly += zapas['goly_domaci']

    pocet_zapasu = len(poslednich_5)
    return vstrelene_goly / pocet_zapasu, obdrzene_goly / pocet_zapasu

In [10]:
data[['domaci_dane_goly', 'domaci_dostane_goly']] = data.apply(lambda row: pd.Series(prum_vstrelenych_obdrzenych(row, data, 'domaci_tym')), axis=1)
data[['hoste_dane_goly', 'hoste_dostane_goly']] = data.apply(lambda row: pd.Series(prum_vstrelenych_obdrzenych(row, data, 'hoste_tym')), axis=1)

In [11]:
def forma_doma_a_venku(row, data, role):
    aktualni_datum = row['datum_zapasu']
    tym = row[role]

    historie = data[data['datum_zapasu'] < aktualni_datum]

    if role == 'domaci_tym':
        zapasy_specificke = historie[historie['domaci_tym'] == tym]
    else:
        zapasy_specificke = historie[historie['hoste_tym'] == tym]

    poslednich_5 = zapasy_specificke.tail(5)

    if len(poslednich_5) == 0:
        return 0.4

    vyhry = 0
    for _, zapas in poslednich_5.iterrows():
        if role == 'domaci_tym' and zapas['goly_domaci'] > zapas['goly_hoste']:
            vyhry += 1
        elif role == 'hoste_tym' and zapas['goly_hoste'] > zapas['goly_domaci']:
            vyhry += 1

    return vyhry / len(poslednich_5)

In [12]:
data['domaci_forma_doma'] = data.apply(lambda row: forma_doma_a_venku(row, data, 'domaci_tym'), axis=1)
data['hoste_forma_venku'] = data.apply(lambda row: forma_doma_a_venku(row, data, 'hoste_tym'), axis=1)

In [13]:
def h2h_vyhry_domacich(row, data):
    aktualni_datum = row['datum_zapasu']
    tym_domaci = row['domaci_tym']
    tym_hoste = row['hoste_tym']

    historie = data[data['datum_zapasu'] < aktualni_datum]

    vzajemne_zapasy = historie[
        ((historie['domaci_tym'] == tym_domaci) & (historie['hoste_tym'] == tym_hoste)) |
        ((historie['domaci_tym'] == tym_hoste) & (historie['hoste_tym'] == tym_domaci))
    ]

    posledni_5 = vzajemne_zapasy.tail(5)

    if len(posledni_5) == 0:
        return 0.5

    vyhry_naseho_domaciho = 0
    for _, zapas in posledni_5.iterrows():
        if zapas['domaci_tym'] == tym_domaci and zapas['goly_domaci'] > zapas['goly_hoste']:
            vyhry_naseho_domaciho += 1
        elif zapas['hoste_tym'] == tym_domaci and zapas['goly_hoste'] > zapas['goly_domaci']:
            vyhry_naseho_domaciho += 1

    return vyhry_naseho_domaciho / len(posledni_5)

In [14]:
data['h2h_uspesnost_domacich'] = data.apply(lambda row: h2h_vyhry_domacich(row, data), axis=1)

In [15]:
def sila_aktualni_sestavy(row, data, role):
    aktualni_datum = row['datum_zapasu']
    hraci_dnes = row[role]
    tym = row['domaci_tym'] if role == 'domaci_hraci' else row['hoste_tym']

    historie = data[data['datum_zapasu'] < aktualni_datum]
    zapasy_tymu = historie[(historie['domaci_tym'] == tym) | (historie['hoste_tym'] == tym)]

    ocekavane_goly_tymu = 0.0

    for hrac in hraci_dnes:
        jmeno_hrace = hrac['jmeno']
        if 'uspesnost_zakroku_procenta' in hrac:
            continue

        goly_hrace_historie = []

        for _, zapas in zapasy_tymu.sort_values('datum_zapasu', ascending=False).iterrows():
            soupiska_tehdy = zapas['domaci_hraci'] if zapas['domaci_tym'] == tym else zapas['hoste_hraci']

            hrac_tehdy = next((p for p in soupiska_tehdy if p['jmeno'] == jmeno_hrace), None)
            if hrac_tehdy:
                goly_hrace_historie.append(hrac_tehdy.get('goly', 0))

            if len(goly_hrace_historie) == 5:
                break

        if goly_hrace_historie:
            ocekavane_goly_tymu += sum(goly_hrace_historie) / len(goly_hrace_historie)

    return ocekavane_goly_tymu

In [16]:
data['domaci_ocekavane_goly'] = data.apply(lambda row: sila_aktualni_sestavy(row, data, 'domaci_hraci'), axis=1)
data['hoste_ocekavane_goly'] = data.apply(lambda row: sila_aktualni_sestavy(row, data, 'hoste_hraci'), axis=1)

In [17]:
def forma_brankare(row, data, role):
    aktualni_datum = row['datum_zapasu']
    hraci_dnes = row[role]
    tym = row['domaci_tym'] if role == 'domaci_hraci' else row['hoste_tym']

    historie = data[data['datum_zapasu'] < aktualni_datum]
    zapasy_tymu = historie[(historie['domaci_tym'] == tym) | (historie['hoste_tym'] == tym)]

    # Najdeme brankáře, který je dnes na soupisce (může jich být víc, vezmeme toho prvního/hlavního)
    brankari_dnes = [h for h in hraci_dnes if 'uspesnost_zakroku_procenta' in h]

    if not brankari_dnes:
        return 5.0

    hlavni_brankar = brankari_dnes[0]['jmeno']

    celkem_zakroku = 0
    celkem_strel = 0
    pocet_zapasu = 0

    for _, zapas in zapasy_tymu.sort_values('datum_zapasu', ascending=False).iterrows():
        soupiska_tehdy = zapas['domaci_hraci'] if zapas['domaci_tym'] == tym else zapas['hoste_hraci']

        brankar_tehdy = next((p for p in soupiska_tehdy if p['jmeno'] == hlavni_brankar), None)
        if brankar_tehdy and 'zakroky' in brankar_tehdy and 'strely_proti' in brankar_tehdy:
            celkem_zakroku += brankar_tehdy['zakroky']
            celkem_strel += brankar_tehdy['strely_proti']
            pocet_zapasu += 1

        if pocet_zapasu == 5:
            break

    if celkem_strel > 0:
        return round((celkem_zakroku / celkem_strel) * 100, 1)
    else:
        return 5.0

In [18]:
data['domaci_forma_brankare'] = data.apply(lambda row: forma_brankare(row, data, 'domaci_hraci'), axis=1)
data['hoste_forma_brankare'] = data.apply(lambda row: forma_brankare(row, data, 'hoste_hraci'), axis=1)

In [19]:
data.tail(10)

,match_id,datum_zapasu,domaci_tym,hoste_tym,goly_domaci,goly_hoste,kurz_domaci,kurz_remiza,kurz_hoste,domaci_hraci,...,domaci_dostane_goly,hoste_dane_goly,hoste_dostane_goly,domaci_forma_doma,hoste_forma_venku,h2h_uspesnost_domacich,domaci_ocekavane_goly,hoste_ocekavane_goly,domaci_forma_brankare,hoste_forma_brankare
2737,14211267,2026-03-15,SC DHfK Leipzig,TVB 1898 Stuttgart,29,29,1.75,8.0,2.70,"[{'jmeno': 'Tomas Mrkva', 'uspesnost_zakroku_p...",...,32.0,32.8,32.2,0.2,0.0,0.60,26.4,34.150000,26.1,19.1
2738,14203131,2026-03-26,HSV Hamburg,Bergischer HC,34,35,1.32,10.0,5.00,"[{'jmeno': 'Azat Valiullin', 'goly': 0}, {'jme...",...,34.0,28.8,30.2,0.4,0.2,0.25,27.2,31.000000,27.8,28.6
2739,14211270,2026-03-27,SG Flensburg Handewitt,TSV GWD Minden,35,31,1.02,25.0,22.00,"[{'jmeno': 'Kent Robin Tonnesen', 'goly': 3}, ...",...,32.0,28.6,34.6,1.0,0.0,1.00,33.2,27.800000,31.7,24.8
2740,14211269,2026-03-27,MT Melsungen,THW Kiel,30,29,3.60,8.5,1.50,"[{'jmeno': 'Kristof Palasics', 'uspesnost_zakr...",...,29.0,29.8,30.4,0.6,0.4,0.60,27.8,34.800000,15.7,25.0
2741,14211272,2026-03-28,SC Magdeburg,Füchse Berlin,35,33,1.78,8.0,2.70,"[{'jmeno': 'Tim Hornke', 'goly': 3}, {'jmeno':...",...,26.0,38.6,28.8,0.8,0.8,0.40,33.0,40.600000,27.3,31.7
2742,14211273,2026-03-28,Rhein Neckar Löwen,HSG Wetzlar,41,27,1.20,12.0,7.50,"[{'jmeno': 'Patrick Groetzki', 'goly': 4}, {'j...",...,29.4,29.4,32.0,0.8,0.2,0.60,32.0,30.600000,24.0,22.7
2743,14211274,2026-03-28,VfL Gummersbach,TSV Hannover Burgdorf,33,27,1.37,9.5,4.50,"[{'jmeno': 'Kentin Mahe', 'goly': 0}, {'jmeno'...",...,26.0,30.2,32.0,0.8,0.4,0.60,32.2,31.400000,13.3,25.9
2744,14203134,2026-03-29,TVB 1898 Stuttgart,TBV Lemgo,32,32,3.50,8.5,1.52,"[{'jmeno': 'Kai Hafner', 'goly': 8}, {'jmeno':...",...,31.8,30.0,28.6,0.8,0.4,0.20,27.8,30.200000,20.6,29.3
2745,14211276,2026-03-29,Frisch Auf Göppingen,HC Erlangen,27,24,1.67,8.0,3.00,"[{'jmeno': 'Marcel Schiller', 'goly': 4}, {'jm...",...,26.2,28.0,32.4,0.2,0.0,0.60,28.6,27.933333,24.2,18.2
2746,14211275,2026-03-29,SC DHfK Leipzig,ThSV Eisenach,29,29,1.80,8.0,2.70,"[{'jmeno': 'Tomas Mrkva', 'uspesnost_zakroku_p...",...,31.8,27.4,29.2,0.2,0.0,0.00,28.0,26.000000,26.0,31.7


In [20]:
data.to_csv('bundesliga_pripravena_data.csv', index=False, encoding='utf-8')